In [1]:
!pip install -q transformers torch

In [2]:
from transformers import pipeline

In [3]:
model_name = 'neuralmind/bert-base-portuguese-cased'
nlp = pipeline('fill-mask', model=model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
texto = "O Brasil é um país [MASK]."

resultados = nlp(texto)

for r in resultados:
  print(f"Sugestão: {r['sequence']} | Score: {r['score']:.4f}")

Sugestão: O Brasil é um país sério. | Score: 0.0873
Sugestão: O Brasil é um país democrático. | Score: 0.0801
Sugestão: O Brasil é um país pequeno. | Score: 0.0411
Sugestão: O Brasil é um país diferente. | Score: 0.0384
Sugestão: O Brasil é um país grande. | Score: 0.0283


In [5]:
!pip install -q datasets

In [6]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer, DataCollatorWithPadding
from datasets import Dataset
import torch

In [16]:
data = {
    "text": [
        "Eu adorei este produto, ele é maravilhoso!",
        "Isso é horrível, nunca mais compro.",
        "O atendimento foi excelente, e rápido.",
        "Estou muito insatisfeito com a qualidade.",
        "Valeu cada centavo, recomento muito.",
        "Péssima experiência, odiei.",
        "O produto superou minhas expectativas, excelente!",
        "Não gostei, veio com defeito.",
        "Funcionou perfeitamente, muito satisfeito.",
        "Muito ruim, qualidade terrível.",
        "Qualidade impecável, voltarei a comprar.",
        "Produto totalmente diferente do anunciado.",
        "Entrega rápida e produto em ótimo estado!",
        "Demorou demais para chegar, péssimo serviço.",
        "Atendimento muito educado e prestativo.",
        "Atendimento horrível, não resolveu nada.",
        "Gostei demais, vale cada centavo.",
        "Experiência muito frustrante, não recomendo."
    ],
    "label": [
        1,0,1,0,1,0, 1,0,1,0,1,0,1,0,1,0, 1, 0  # Novos dados adicionados
    ]
}

dataset = Dataset.from_dict(data)

In [8]:
model_checkpoint = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. C

In [17]:
def tokenizer_function(examples):
  return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenizer_dataset = dataset.map(tokenizer_function, batched=True)
tokenizer_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

Map:   0%|          | 0/18 [00:00<?, ? examples/s]

In [18]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    logging_steps=1,
    eval_strategy="no",
    report_to="none"
)

In [19]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenizer_dataset,
    data_collator=data_collator
)

In [20]:
print("Iniciando fine-tuning corrigido...")
trainer.train()
print("Treinamento concluído")

Iniciando fine-tuning corrigido...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,0.797570
2,0.265425
3,0.748585
4,0.618266
5,0.468047
6,0.150829
7,0.187728
8,0.356232
9,0.074607
10,0.063943


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Treinamento concluído


In [21]:
frase_test = "O produto péssimo, de má qualidade, odiei!"
inputs = tokenizer(frase_test, return_tensors="pt", truncation=True, padding=True, max_length=128)

with torch.no_grad():
  logits = model(**inputs).logits

predicao = torch.argmax(logits, dim=1).item()
sentimento = "Positivo" if predicao == 1 else "Negativo"

print(f"Frase: {frase_test}")
print(f"Sentimento previsto: {sentimento}")

Frase: O produto péssimo, de má qualidade, odiei!
Sentimento previsto: Negativo
